In [ ]:
!pip install -r requirements.txt

In [1]:
!pip list

Package                   Version
------------------------- ------------
absl-py                   2.3.1
accelerate                1.9.0
aiohappyeyeballs          2.6.1
aiohttp                   3.12.15
aiosignal                 1.4.0
anyio                     4.7.0
argon2-cffi               21.3.0
argon2-cffi-bindings      21.2.0
asttokens                 3.0.0
async-lru                 2.0.4
attrs                     25.3.0
babel                     2.16.0
beautifulsoup4            4.13.4
bitsandbytes              0.46.1
bleach                    6.2.0
brotlicffi                1.0.9.2
certifi                   2025.7.14
cffi                      1.17.1
charset-normalizer        3.4.2
comm                      0.2.1
contourpy                 1.3.3
cycler                    0.12.1
datasets                  4.0.0
debugpy                   1.8.11
decorator                 5.1.1
defusedxml                0.7.1
dill                      0.3.8
einops                    0.8.1
executing     

In [2]:
# Import .env file into os env variables
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
!huggingface-cli login --token $HF_TOKEN --add-to-git-credential

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
Token is valid (permission: write).
The token `full_access` has been saved to /home/sobhan/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (manager).
Your token has been saved to /home/sobhan/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,expandable_segments:True"

import torch
import gc

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU Available: True
GPU Name: NVIDIA GeForce RTX 3090
GPU Memory: 23.5 GB


In [5]:
from dataset_builder import MixtureDataBuilder
from llm_layer_pruner import LLMLayerPruner

# ---------------------------
# 1. Setup
# ---------------------------
model_name = "Qwen/Qwen2.5-1.5B"   # small for testing
pruner = LLMLayerPruner(
    model_name=model_name,
    default_percentages=[10, 20, 30, 40, 50],  # unified default percentages
    default_rank=16,
    default_max_heal_steps=500,
    default_max_qlora_steps=1000,
    blank_noise_std=0.01,
)

# Build separate Train/Val/Test mixtures with token budgets per domain:
builder = MixtureDataBuilder(pruner.tokenizer, max_length=512, seed=42, verbose=True)

token_targets_by_split = {
    "train": {"syntax": 200_000, "code": 200_000, "math": 200_000},
    "test": {"syntax": 50_000, "code": 50_000, "math": 50_000},
}

texts_by_split, lengths_by_split, summary_by_split = builder.build_split_mixtures(
    token_targets_by_split,
    val_from_train_when_missing=True,
    # order_of_splits=["test","validation","train"]  # optional, this is the default
)

# Make dataloaders (you can omit a split by leaving it out of token_targets_by_split)
loaders = builder.make_dataloaders_for_splits(
    texts_by_split,
    batch_sizes={"test": 8, "train": 2},
    num_workers=4,
    pin_memory=True,
    shuffle=False,
)

train_dl = loaders["train"]
eval_dl = loaders["test"]


# Path to your analysis CSVs from earlier steps
agg_csvs = {
    "weighted_mean": f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_weighted_mean.csv",
    # "minimax":       f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_minimax.csv",
}

# ---------------------------
# 2. Pre-heal pruning sweep
# ---------------------------
df_pre = pruner.run_single_shot_pruning_sweep(
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    train_dataloader=train_dl,
    strategies=["none", "tblock_blank", "tblock_avg", "tblock_lr_mean", "tblock_consensus"],
)
display(df_pre)

# ---------------------------
# 3. Single-layer heal sweep
# ---------------------------
df_heal = pruner.run_single_layer_heal_sweep(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    strategies=["tblock_blank", "tblock_avg", "tblock_lr_mean", "tblock_consensus"],
    lr=2e-4,
    max_steps=350,
    grad_accum=4,
)
display(df_heal)

# ---------------------------
# 4. Full-model QLoRA healing experiments
# ---------------------------
df_full = pruner.run_full_heal_experiments(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    qlora_lr=1e-4,
    max_qlora_steps=700,
    grad_accum=4,
)
display(df_full)

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Mixture] Fallback for domain 'code': requested 'test' but using 'train'
[Mixture] Built 'test': domains={'syntax': 50374, 'code': 50084, 'math': 50032} | used_splits={'syntax': 'test', 'code': 'train', 'math': 'test'}
[Mixture] Built 'train': domains={'syntax': 200046, 'code': 200126, 'math': 200034} | used_splits={'syntax': 'train', 'code': 'train', 'math': 'train'}


Perplexity: 100%|██████████| 164/164 [00:11<00:00, 14.22it/s]


Baseline PPL: 12.409 (layers=28, hidden=1536)


Perplexity: 100%|██████████| 164/164 [00:06<00:00, 24.35it/s]


...

[Single-layer heal] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_single_layer_heal.csv


,method,strategy,percent,selected_start,selected_end_inclusive,ppl_before_heal,ppl_after_heal,delta_ppl,inserted_idx,rank
0,weighted_mean,tblock_blank,10,12,15,12.194977,10.285704,-1.909273,12,16
1,weighted_mean,tblock_avg,10,12,15,12.123243,9.806059,-2.317183,12,16
2,weighted_mean,tblock_lr_mean,10,12,15,12.252200,10.084854,-2.167346,12,16
3,weighted_mean,tblock_consensus,10,12,15,12.221641,10.199508,-2.022133,12,16
4,weighted_mean,tblock_blank,20,9,15,16.040251,11.240519,-4.799731,9,16
5,weighted_mean,tblock_avg,20,9,15,15.765839,10.740537,-5.025301,9,16
6,weighted_mean,tblock_lr_mean,20,9,15,16.073602,11.269462,-4.804140,9,16
7,weighted_mean,tblock_consensus,20,9,15,16.051596,11.295673,-4.755923,9,16
8,weighted_mean,tblock_blank,30,8,16,19.716346,12.652171,-7.064176,8,16
9,weighted_mean,tblock_avg,30,8,16,19.411582,11.711585,-7.699997,8,16


Perplexity: 100%|██████████| 164/164 [00:17<00:00,  9.37it/s]


[Full-heal QLoRA] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_full_heal_qlora.csv


,variant,percent,method,ppl_pre_qlora,ppl_post_qlora,adapters,ok,error
0,none,10,weighted_mean,11.563365,13.585173,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
1,tblock_blank_preheal,10,weighted_mean,12.194977,10.085325,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
2,tblock_avg_post_single_layer_heal,10,weighted_mean,9.806059,10.095195,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
3,none,20,weighted_mean,13.710756,11.611507,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
4,tblock_blank_preheal,20,weighted_mean,16.040251,13.181382,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
5,tblock_avg_post_single_layer_heal,20,weighted_mean,10.740537,13.138510,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
6,none,30,weighted_mean,17.134082,14.730461,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
7,tblock_blank_preheal,30,weighted_mean,19.716346,17.105916,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
8,tblock_avg_post_single_layer_heal,30,weighted_mean,11.711585,14.815254,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
9,none,40,weighted_mean,30.934981,16.829581,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None


In [5]:
from transformers import BitsAndBytesConfig
from llm_layer_pruner import LLMLayerPruner
from dataset_builder import MixtureDataBuilder

# ---------------------------
# 1. Setup
# ---------------------------

# Load the model with 8-bit quantization
bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,                    # int8 linear layers
)

# Load the model with 4-bit quantization
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # QLoRA default
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    ),
)

# Load the model
model_name = "Qwen/Qwen2.5-1.5B"
pruner = LLMLayerPruner(
    model_name=model_name,
    # quant_config=bnb_4bit,
    default_percentages=[10, 20], #, 30, 40, 50],
    default_rank=16,
    default_max_heal_steps=500,
    default_max_qlora_steps=1000,
    blank_noise_std=0.01,                 # small noise for the blank layer
)

pruner.configure_profiles(
    ppl=None,
    single_heal=bnb_8bit,
    qlora=bnb_4bit,
    max_eval_seq_len=512,
)

builder = MixtureDataBuilder(pruner.tokenizer, max_length=512, seed=42, verbose=True)

# Build separate Train/Val/Test mixtures with token budgets per domain:
token_targets_by_split = {
    "train": {"syntax": 20_000, "code": 20_000, "math": 20_000},
    "test":  {"syntax": 5_000, "code": 5_000, "math": 5_000},
}

texts_by_split, lengths_by_split, summary_by_split = builder.build_split_mixtures(
    token_targets_by_split,
    val_from_train_when_missing=True
)


# Make dataloaders (you can omit a split by leaving it out of token_targets_by_split)
loaders = builder.make_dataloaders_for_splits(
    texts_by_split,
    batch_sizes={ "train": 2, "test": 8},
    num_workers=4,
    pin_memory=True,
    shuffle=False,
)

train_dl = loaders["train"]
eval_dl = loaders["test"]


# Path to your analysis CSVs from earlier steps
agg_csvs = {
    "weighted_mean": f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_weighted_mean.csv",
    # "minimax":       f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_minimax.csv",
}

# ---------------------------
# 2. Pre-heal pruning sweep
# ---------------------------
df_pre = pruner.run_single_shot_pruning_sweep(
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    train_dataloader=train_dl,
    strategies=["none", "tblock_blank"], #, "tblock_avg", "tblock_lr_mean", "tblock_consensus"],
)
display(df_pre)

# ---------------------------
# 3. Single-layer heal sweep
# ---------------------------
df_heal = pruner.run_single_layer_heal_sweep(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    strategies=["tblock_blank", "tblock_avg"], #, "tblock_lr_mean", "tblock_consensus"],
    lr=2e-4,
    max_steps=350,
    grad_accum=4,
)
display(df_heal)

# ---------------------------
# 4. Full-model QLoRA healing experiments
# ---------------------------
df_full = pruner.run_full_heal_experiments(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    qlora_lr=1e-4,
    max_qlora_steps=700,
    grad_accum=4,
)
display(df_full)

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Mixture] Fallback for domain 'code': requested 'test' but using 'train'
[Mixture] Built 'test': domains={'syntax': 5081, 'code': 5326, 'math': 5010} | used_splits={'syntax': 'test', 'code': 'train', 'math': 'test'}
[Mixture] Built 'train': domains={'syntax': 20018, 'code': 20158, 'math': 20000} | used_splits={'syntax': 'train', 'code': 'train', 'math': 'train'}


Perplexity: 100%|██████████| 17/17 [00:01<00:00,  8.64it/s]


Baseline PPL: 12.176 (layers=28, hidden=1536)


Perplexity: 100%|██████████| 17/17 [00:01<00:00,  9.94it/s]


[Pre-heal sweep] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_preheal_sweep.csv


,method,strategy,percent,selected_start,selected_end_inclusive,baseline_ppl,post_ppl,delta_ppl,inserted_idx,rank
0,weighted_mean,none,10,13,15,12.175613,11.489751,-0.685862,NaN,16
1,weighted_mean,tblock_blank,10,12,15,12.175613,12.251654,0.076041,12.0,16
2,weighted_mean,none,20,11,16,12.175613,13.919271,1.743658,NaN,16
3,weighted_mean,tblock_blank,20,9,15,12.175613,16.340384,4.164771,9.0,16


Perplexity: 100%|██████████| 17/17 [00:01<00:00,  9.31it/s]


UnboundLocalError: cannot access local variable 'm' where it is not associated with a value

In [5]:
from transformers import BitsAndBytesConfig
from llm_layer_pruner import LLMLayerPruner
from dataset_builder import MixtureDataBuilder

# ---------------------------
# 1. Setup
# ---------------------------

# Load the model with 8-bit quantization
bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,                    # int8 linear layers
)

# Load the model with 4-bit quantization
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # QLoRA default
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    ),
)

# Load the model
model_name = "Qwen/Qwen2.5-1.5B"
pruner = LLMLayerPruner(
    model_name=model_name,
    # quant_config=bnb_4bit,
    default_percentages=[10, 20, 30, 40, 50],
    default_rank=16,
    default_max_heal_steps=500,
    default_max_qlora_steps=1000,
    blank_noise_std=0.01,                 # small noise for the blank layer
)

pruner.configure_profiles(
    ppl=None,
    single_heal=bnb_8bit,
    qlora=bnb_4bit,
    max_eval_seq_len=512,
)

builder = MixtureDataBuilder(pruner.tokenizer, max_length=512, seed=42, verbose=True)

# Build separate Train/Val/Test mixtures with token budgets per domain:
token_targets_by_split = {
    "train": {"syntax": 200_000, "code": 200_000, "math": 200_000},
    "test":  {"syntax": 50_000, "code": 50_000, "math": 50_000},
}

texts_by_split, lengths_by_split, summary_by_split = builder.build_split_mixtures(
    token_targets_by_split,
    val_from_train_when_missing=True
)


# Make dataloaders (you can omit a split by leaving it out of token_targets_by_split)
loaders = builder.make_dataloaders_for_splits(
    texts_by_split,
    batch_sizes={ "train": 2, "test": 8},
    num_workers=4,
    pin_memory=True,
    shuffle=False,
)

train_dl = loaders["train"]
eval_dl = loaders["test"]


# Path to your analysis CSVs from earlier steps
agg_csvs = {
    "weighted_mean": f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_weighted_mean.csv",
    # "minimax":       f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_minimax.csv",
}

# ---------------------------
# 2. Pre-heal pruning sweep
# ---------------------------
df_pre = pruner.run_single_shot_pruning_sweep(
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    train_dataloader=train_dl,
    strategies=["none", "tblock_blank", "tblock_avg", "tblock_lr_mean", "tblock_consensus"],
)
display(df_pre)

# ---------------------------
# 3. Single-layer heal sweep
# ---------------------------
df_heal = pruner.run_single_layer_heal_sweep(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    strategies=["tblock_blank", "tblock_avg", "tblock_lr_mean", "tblock_consensus"],
    lr=2e-4,
    max_steps=350,
    grad_accum=4,
)
display(df_heal)

# ---------------------------
# 4. Full-model QLoRA healing experiments
# ---------------------------
df_full = pruner.run_full_heal_experiments(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    qlora_lr=1e-4,
    max_qlora_steps=700,
    grad_accum=4,
)
display(df_full)

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Mixture] Fallback for domain 'code': requested 'test' but using 'train'
[Mixture] Built 'test': domains={'syntax': 50374, 'code': 50084, 'math': 50032} | used_splits={'syntax': 'test', 'code': 'train', 'math': 'test'}
[Mixture] Built 'train': domains={'syntax': 200046, 'code': 200126, 'math': 200034} | used_splits={'syntax': 'train', 'code': 'train', 'math': 'train'}


Perplexity: 100%|██████████| 164/164 [00:11<00:00, 13.67it/s]


Baseline PPL: 12.409 (layers=28, hidden=1536)


Perplexity: 100%|██████████| 164/164 [00:07<00:00, 22.57it/s]


[Pre-heal sweep] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_preheal_sweep.csv


,method,strategy,percent,selected_start,selected_end_inclusive,baseline_ppl,post_ppl,delta_ppl,inserted_idx,rank
0,weighted_mean,none,10,13,15,12.408719,11.563365,-0.845354,NaN,16
1,weighted_mean,tblock_blank,10,12,15,12.408719,12.192314,-0.216405,12.0,16
2,weighted_mean,tblock_avg,10,12,15,12.408719,12.123243,-0.285476,12.0,16
3,weighted_mean,tblock_lr_mean,10,12,15,12.408719,12.252200,-0.156519,12.0,16
4,weighted_mean,tblock_consensus,10,12,15,12.408719,12.221641,-0.187079,12.0,16
5,weighted_mean,none,20,11,16,12.408719,13.710756,1.302037,NaN,16
6,weighted_mean,tblock_blank,20,9,15,12.408719,16.041728,3.633009,9.0,16
7,weighted_mean,tblock_avg,20,9,15,12.408719,15.765839,3.357119,9.0,16
8,weighted_mean,tblock_lr_mean,20,9,15,12.408719,16.073602,3.664883,9.0,16
9,weighted_mean,tblock_consensus,20,9,15,12.408719,16.051596,3.642877,9.0,16


Heal@layer12:  18%|█▊        | 479/2607 [01:09<05:10,  6.84it/s]


KeyboardInterrupt: 

In [5]:
from transformers import BitsAndBytesConfig
from llm_layer_pruner import LLMLayerPruner
from dataset_builder import MixtureDataBuilder

# ---------------------------
# 1. Setup
# ---------------------------

# Load the model with 8-bit quantization
bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,                    # int8 linear layers
)

# Load the model with 4-bit quantization
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # QLoRA default
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    ),
)

# Load the model
model_name = "Qwen/Qwen2.5-1.5B"
pruner = LLMLayerPruner(
    model_name=model_name,
    # quant_config=bnb_4bit,
    default_percentages=[10, 20], #, 30, 40, 50],
    default_rank=16,
    default_max_heal_steps=500,
    default_max_qlora_steps=1000,
    blank_noise_std=0.01,                 # small noise for the blank layer
)

pruner.configure_profiles(
    ppl=None,
    single_heal=None,
    qlora=None,
    max_eval_seq_len=512,
)

builder = MixtureDataBuilder(pruner.tokenizer, max_length=512, seed=42, verbose=True)

# Build separate Train/Val/Test mixtures with token budgets per domain:
token_targets_by_split = {
    "train": {"syntax": 200_000, "code": 200_000, "math": 200_000},
    "test":  {"syntax": 50_000, "code": 50_000, "math": 50_000},
}

texts_by_split, lengths_by_split, summary_by_split = builder.build_split_mixtures(
    token_targets_by_split,
    val_from_train_when_missing=True
)


# Make dataloaders (you can omit a split by leaving it out of token_targets_by_split)
loaders = builder.make_dataloaders_for_splits(
    texts_by_split,
    batch_sizes={ "train": 2, "test": 8},
    num_workers=4,
    pin_memory=True,
    shuffle=False,
)

train_dl = loaders["train"]
eval_dl = loaders["test"]


# Path to your analysis CSVs from earlier steps
agg_csvs = {
    "weighted_mean": f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_weighted_mean.csv",
    # "minimax":       f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_minimax.csv",
}

# ---------------------------
# 2. Pre-heal pruning sweep
# ---------------------------
df_pre = pruner.run_single_shot_pruning_sweep(
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    train_dataloader=train_dl,
    strategies=["none", "tblock_blank", "tblock_avg"], #, "tblock_lr_mean", "tblock_consensus"],
)
display(df_pre)

# ---------------------------
# 3. Single-layer heal sweep
# ---------------------------
df_heal = pruner.run_single_layer_heal_sweep(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    strategies=["tblock_blank", "tblock_avg"], #, "tblock_lr_mean", "tblock_consensus"],
    lr=2e-4,
    max_steps=350,
    grad_accum=4,
)
display(df_heal)

# ---------------------------
# 4. Full-model QLoRA healing experiments
# ---------------------------
df_full = pruner.run_full_heal_experiments(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    qlora_lr=1e-4,
    max_qlora_steps=700,
    grad_accum=4,
)
display(df_full)

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Mixture] Fallback for domain 'code': requested 'test' but using 'train'
[Mixture] Built 'test': domains={'syntax': 50374, 'code': 50084, 'math': 50032} | used_splits={'syntax': 'test', 'code': 'train', 'math': 'test'}
[Mixture] Built 'train': domains={'syntax': 200046, 'code': 200126, 'math': 200034} | used_splits={'syntax': 'train', 'code': 'train', 'math': 'train'}


Perplexity: 100%|██████████| 164/164 [00:12<00:00, 13.52it/s]


Baseline PPL: 12.409 (layers=28, hidden=1536)


Perplexity: 100%|██████████| 164/164 [00:09<00:00, 16.84it/s]


...

[Single-layer heal] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_single_layer_heal.csv


,method,strategy,percent,selected_start,selected_end_inclusive,ppl_before_heal,ppl_after_heal,delta_ppl,inserted_idx,rank
0,weighted_mean,tblock_blank,10,12,15,12.193295,10.342245,-1.851050,12,16
1,weighted_mean,tblock_avg,10,12,15,12.123243,9.812681,-2.310562,12,16
2,weighted_mean,tblock_blank,20,9,15,16.040329,11.409420,-4.630910,9,16
3,weighted_mean,tblock_avg,20,9,15,15.765839,10.740783,-5.025055,9,16


Perplexity: 100%|██████████| 164/164 [00:09<00:00, 16.43it/s]


[Full-heal QLoRA] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_full_heal_qlora.csv


,variant,percent,method,ppl_pre_qlora,ppl_post_qlora,adapters,ok,error
0,none,10,weighted_mean,11.563365,9.777906,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
1,tblock_blank_preheal,10,weighted_mean,12.193295,10.452549,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
2,tblock_avg_post_single_layer_heal,10,weighted_mean,9.812681,10.472776,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
3,none,20,weighted_mean,13.710756,12.858397,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
4,tblock_blank_preheal,20,weighted_mean,16.040329,15.154206,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
5,tblock_avg_post_single_layer_heal,20,weighted_mean,10.740783,14.289465,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None


In [5]:
from transformers import BitsAndBytesConfig
from llm_layer_pruner import LLMLayerPruner
from dataset_builder import MixtureDataBuilder

# ---------------------------
# 1. Setup
# ---------------------------

# Load the model with 8-bit quantization
bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,                    # int8 linear layers
)

# Load the model with 4-bit quantization
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # QLoRA default
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    ),
)

# Load the model
model_name = "Qwen/Qwen2.5-1.5B"
pruner = LLMLayerPruner(
    model_name=model_name,
    # quant_config=bnb_4bit,
    default_percentages=[10, 20], #, 30, 40, 50],
    default_rank=16,
    default_max_heal_steps=500,
    default_max_qlora_steps=1000,
    blank_noise_std=0.01,                 # small noise for the blank layer
)

pruner.configure_profiles(
    ppl=None,
    single_heal=None,
    qlora=None,
    max_eval_seq_len=512,
)

builder = MixtureDataBuilder(pruner.tokenizer, max_length=512, seed=42, verbose=True)

# Build separate Train/Val/Test mixtures with token budgets per domain:
token_targets_by_split = {
    "train": {"syntax": 200_000, "code": 200_000, "math": 200_000},
    "test":  {"syntax": 50_000, "code": 50_000, "math": 50_000},
}

texts_by_split, lengths_by_split, summary_by_split = builder.build_split_mixtures(
    token_targets_by_split,
    val_from_train_when_missing=True
)


# Make dataloaders (you can omit a split by leaving it out of token_targets_by_split)
loaders = builder.make_dataloaders_for_splits(
    texts_by_split,
    batch_sizes={ "train": 2, "test": 8},
    num_workers=4,
    pin_memory=True,
    shuffle=False,
)

train_dl = loaders["train"]
eval_dl = loaders["test"]


# Path to your analysis CSVs from earlier steps
agg_csvs = {
    "weighted_mean": f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_weighted_mean.csv",
    # "minimax":       f"results/analysis/{pruner._slugify(model_name)}/{pruner._slugify(model_name)}_aggregate_minimax.csv",
}

# ---------------------------
# 2. Pre-heal pruning sweep
# ---------------------------
df_pre = pruner.run_single_shot_pruning_sweep(
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    train_dataloader=train_dl,
    strategies=["none", "tblock_blank", "tblock_avg"], #, "tblock_lr_mean", "tblock_consensus"],
)
display(df_pre)

# ---------------------------
# 3. Single-layer heal sweep
# ---------------------------
df_heal = pruner.run_single_layer_heal_sweep(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    strategies=["tblock_blank", "tblock_avg"], #, "tblock_lr_mean", "tblock_consensus"],
    lr=2e-4,
    max_steps=350,
    grad_accum=4,
)
display(df_heal)

# ---------------------------
# 4. Full-model QLoRA healing experiments
# ---------------------------
df_full = pruner.run_full_heal_experiments(
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    agg_method_csvs=agg_csvs,
    qlora_lr=1e-4,
    max_qlora_steps=700,
    grad_accum=4,
)
display(df_full)

/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sobhan/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Mixture] Fallback for domain 'code': requested 'test' but using 'train'
[Mixture] Built 'test': domains={'syntax': 50374, 'code': 50084, 'math': 50032} | used_splits={'syntax': 'test', 'code': 'train', 'math': 'test'}
[Mixture] Built 'train': domains={'syntax': 200046, 'code': 200126, 'math': 200034} | used_splits={'syntax': 'train', 'code': 'train', 'math': 'train'}


Perplexity: 100%|██████████| 164/164 [00:12<00:00, 13.42it/s]


Baseline PPL: 12.409 (layers=28, hidden=1536)


Perplexity: 100%|██████████| 164/164 [00:09<00:00, 16.82it/s]


...

[Single-layer heal] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_single_layer_heal.csv


,method,strategy,percent,selected_start,selected_end_inclusive,ppl_before_heal,ppl_after_heal,delta_ppl,inserted_idx,rank
0,weighted_mean,tblock_blank,10,12,15,12.195708,10.138438,-2.057270,12,16
1,weighted_mean,tblock_avg,10,12,15,12.123243,9.801807,-2.321435,12,16
2,weighted_mean,tblock_blank,20,9,15,16.039675,11.351434,-4.688241,9,16
3,weighted_mean,tblock_avg,20,9,15,15.765839,10.749293,-5.016546,9,16


Perplexity: 100%|██████████| 164/164 [00:10<00:00, 16.28it/s]


[Full-heal QLoRA] Saved: results/pruning/single-shot/qwen-qwen2.5-1.5b/qwen-qwen2.5-1.5b_full_heal_qlora.csv


,variant,percent,method,ppl_pre_qlora,ppl_post_qlora,adapters,ok,error
0,none,10,weighted_mean,11.563365,9.883399,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
1,tblock_blank_preheal,10,weighted_mean,12.195708,10.461172,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
2,tblock_avg_post_single_layer_heal,10,weighted_mean,9.801807,10.405285,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
3,none,20,weighted_mean,13.710756,12.901056,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
4,tblock_blank_preheal,20,weighted_mean,16.039675,14.358333,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
5,tblock_avg_post_single_layer_heal,20,weighted_mean,10.749293,14.199004,results/pruning/single-shot/qwen-qwen2.5-1.5b/...,True,None
